In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from pm.utils.data import DataManager

from utils.components import (
    MacroDataDownloader,
    MacroTransformCalculator,
    MacroCorrelationCalculator,
    MacroRegressionCalculator,
    FactorCollinearityAnalyzer,
)

from utils.analyzers import (
    MacroFactorAnalyzer,
    MacroCorrelationAnalyzer,
    MacroSensitivityAnalyzer,
    MacroSituationAnalyzer,
    FactorSelectionAnalyzer
)

from utils.reporters import (
    MacroFactorReporter,
    MacroCorrelationReporter,
    MacroSensitivityReporter,
    MacroSituationReporter
)

from utils.visualizations import (
    MacroFactorVisualizer,
    MacroCorrelationVisualizer,
    MacroSensitivityVisualizer,
    MacroSituationVisualizer,
)

from utils.tools import (
    MACRO_GLOBAL_FACTORS,
    FACTORS_TO_USE,
    ANNUAL_FACTOR,   
    REGIME_WINDOW,
    MAX_LAG,
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# LOW-LEVEL COMPONENTS (Specific calculations)
# ═══════════════════════════════════════════════════════════════════

# Macro data downloader
data_downloader = MacroDataDownloader()

# Factor transformer
transform_calc = MacroTransformCalculator()

# Specific calculators
correlation_calc = MacroCorrelationCalculator(max_lag=MAX_LAG)
regression_calc = MacroRegressionCalculator(annual_factor=ANNUAL_FACTOR, use_hac=True)
collinearity_calc = FactorCollinearityAnalyzer(corr_threshold=0.9, vif_threshold=10.0)

# ═══════════════════════════════════════════════════════════════════
# HIGH-LEVEL ANALYZERS (Orchestration + Insights)
# ═══════════════════════════════════════════════════════════════════

factor_analyzer = MacroFactorAnalyzer(annual_factor=ANNUAL_FACTOR)
corr_analyzer = MacroCorrelationAnalyzer(max_lag=MAX_LAG)
sens_analyzer = MacroSensitivityAnalyzer(annual_factor=ANNUAL_FACTOR)
situation_analyzer = MacroSituationAnalyzer()
selector = FactorSelectionAnalyzer(
    corr_threshold=0.9, 
    vif_threshold=10.0, 
    force_keep=FACTORS_TO_USE
)

# ═══════════════════════════════════════════════════════════════════
# REPORTERS & VISUALIZERS
# ═══════════════════════════════════════════════════════════════════

factor_reporter = MacroFactorReporter(factor_analyzer)
corr_reporter = MacroCorrelationReporter(corr_analyzer)
sens_reporter = MacroSensitivityReporter(sens_analyzer)
situation_reporter = MacroSituationReporter(situation_analyzer)

factor_viz = MacroFactorVisualizer(factor_analyzer)
corr_viz = MacroCorrelationVisualizer(corr_analyzer)
sens_viz = MacroSensitivityVisualizer(sens_analyzer)
situation_viz = MacroSituationVisualizer(situation_analyzer)

In [ ]:
# Portfolio configuration
TICKERS = ["BTC-EUR", "IONQ", "META"]
WEIGHTS = np.ones(len(TICKERS)) / len(TICKERS)
START_DATE = "2020-01-01"
END_DATE = "2024-12-31"

print(f"Portfolio: {', '.join(TICKERS)}")
print(f"Weights: Equal-weighted ({WEIGHTS[0]:.2%} each)")
print(f"Period: {START_DATE} -> {END_DATE}")

In [ ]:
# Download portfolio data
data_manager = DataManager()
assets_prices, _ = data_manager.download_portfolio_with_benchmark(
    tickers=TICKERS,
    benchmark_name="SP500",
    start_date=START_DATE,
    end_date=END_DATE
)

# Calculate portfolio returns
returns = np.log(assets_prices / assets_prices.shift(1)).dropna()
portfolio_returns = (returns * WEIGHTS).sum(axis=1)

print(f"\n[OK] Portfolio returns: {len(portfolio_returns)} observations")

# Validate data before continuing
if len(portfolio_returns) == 0:
    print("[ERROR] Could not obtain portfolio data.")
    print("   Verify that all tickers are valid and have available data.")
    print(f"   Requested tickers: {TICKERS}")
    raise ValueError("Empty portfolio: cannot calculate returns")

# Safe to access indices now
print(f"   Range: {portfolio_returns.index[0].date()} -> {portfolio_returns.index[-1].date()}")
print(f"   Average daily return: {portfolio_returns.mean():.4%}")
print(f"   Daily volatility: {portfolio_returns.std():.4%}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 1: DOWNLOAD MACRO FACTORS
# ═══════════════════════════════════════════════════════════════════

print("Downloading macro factors...")
factors_raw = data_downloader.download_factors(
    factor_names=MACRO_GLOBAL_FACTORS, 
    start_date=START_DATE,
    end_date=END_DATE,
    progress=True
)
print(f"[OK] Downloaded {len(factors_raw)} factors")

# ═══════════════════════════════════════════════════════════════════
# STEP 2: FACTOR TRANSFORMATION
# ═══════════════════════════════════════════════════════════════════

print("\nTransforming factors (log returns, diffs, scaling)...")
factors_dict, factors_df = transform_calc.transform_all_factors(
    factors_raw,
    target_index=portfolio_returns.index,
    fill_method='ffill'
)
print(f"[OK] {factors_df.shape[1]} factors transformed")

# ═══════════════════════════════════════════════════════════════════
# STEP 3: SPREAD CALCULATION
# ═══════════════════════════════════════════════════════════════════

print("\nCalculating spreads...")
spreads_df = transform_calc.calculate_all_spreads(factors_df)

if not spreads_df.empty:
    print(f"[OK] {spreads_df.shape[1]} spreads calculated:")
    for spread in spreads_df.columns:
        print(f"   - {spread}")
    factors_complete = pd.concat([factors_df, spreads_df], axis=1)
else:
    print("[!] No spreads calculated")
    factors_complete = factors_df.copy()

# ═══════════════════════════════════════════════════════════════════
# STEP 4: ALIGNMENT WITH PORTFOLIO
# ═══════════════════════════════════════════════════════════════════

print("\nAligning factors with portfolio...")
factors_aligned = transform_calc.align_to_portfolio(
    factors_complete, 
    portfolio_returns
)

# Select specific factors + spreads
factors_to_analyze = FACTORS_TO_USE + list(spreads_df.columns)
factors_for_analysis = factors_aligned[factors_to_analyze].dropna()

print(f"[OK] Factors ready for analysis:")
print(f"   - Factors: {factors_for_analysis.shape[1]}")
print(f"   - Observations: {factors_for_analysis.shape[0]}")
print(f"   - Period: {factors_for_analysis.index[0].date()} -> {factors_for_analysis.index[-1].date()}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# COLLINEARITY CLEANUP
# ═══════════════════════════════════════════════════════════════════

print("Analyzing collinearity between factors...")
sel_res = selector.analyze(factors_for_analysis)
pruned_factors = sel_res["pruned_factors"]

print(f"\nCleanup results:")
print(f"   - Original factors: {factors_for_analysis.shape[1]}")
print(f"   - Factors after cleanup: {pruned_factors.shape[1]}")
print(f"   - Factors removed: {factors_for_analysis.shape[1] - pruned_factors.shape[1]}")

if factors_for_analysis.shape[1] - pruned_factors.shape[1] > 0:
    removed = set(factors_for_analysis.columns) - set(pruned_factors.columns)
    print(f"   - Removed due to collinearity: {', '.join(removed)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MACRO FACTOR ANALYSIS
# ═══════════════════════════════════════════════════════════════════

print("Running multifactor regression...")
factor_results = factor_analyzer.analyze(
    portfolio_returns,
    pruned_factors,
    use_hac=True
)

# Detailed report
factor_reporter.print_analysis(factor_results)

# Visualization
fig = factor_viz.plot_factor_analysis(factor_results, top_n=10)
plt.show()

print("\nInsight: Check R-squared and significant factors")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# LAGGED CORRELATION ANALYSIS
# ═══════════════════════════════════════════════════════════════════

print("Analyzing correlations with lags...")
corr_results = corr_analyzer.analyze(
    portfolio_returns,
    factors_for_analysis
)

# Detailed report
corr_reporter.print_analysis(corr_results)

# Visualization
fig = corr_viz.plot_correlation_analysis(corr_results, top_n=10)
plt.show()

print("\nInsight: Leading factors predict future portfolio movements")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SENSITIVITY ANALYSIS
# ═══════════════════════════════════════════════════════════════════

print("Analyzing sensitivities to macro factors...")
sens_results = sens_analyzer.analyze(
    portfolio_returns,
    factors_for_analysis
)

# Report
sens_reporter.print_analysis(sens_results)

# Rolling analysis
print("\nCalculating rolling betas...")
rolling_betas = sens_analyzer.analyze_rolling(
    portfolio_returns,
    factors_for_analysis,
    window=REGIME_WINDOW
)

# Combined visualization
fig = sens_viz.plot_sensitivity_analysis(
    sens_results, 
    rolling_betas=rolling_betas
)
plt.show()

print("\nInsight: Sensitivities can change over time (see rolling betas)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SENSITIVITY EVOLUTION OVER TIME
# ═══════════════════════════════════════════════════════════════════

fig = sens_viz.plot_rolling_betas(rolling_betas, figsize=(16, 10))
plt.show()

print("Insight: Identify regime changes in factor exposures")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MACROECONOMIC SITUATION ANALYSIS
# ═══════════════════════════════════════════════════════════════════

print("Analyzing global macroeconomic situation...")
situation_results = situation_analyzer.analyze(factors_raw)

# Detailed report
situation_reporter.print_situation(situation_results)

# Visual dashboard
fig = situation_viz.plot_macro_situation(situation_results)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# DIRECT COMPONENT ANALYSIS (Optional - More Control)
# ═══════════════════════════════════════════════════════════════════

# Example 1: Correlation with specific lag for a factor
print("Detailed analysis: SP500 correlation with lags")
sp500_corr = correlation_calc.calculate_lagged(
    y=portfolio_returns,
    x=factors_for_analysis['SP500']
)
print(sp500_corr.nlargest(5, 'corr'))

# Example 2: Regression with only 2 factors
print("\nSimplified analysis: SP500 and VIX only")
simple_factors = factors_for_analysis[['SP500', 'VIX']]
simple_regression = regression_calc.calculate_multifactor(
    portfolio_returns,
    simple_factors
)
print(f"Simple model R-squared: {simple_regression.r_squared:.3f}")
print(f"Beta SP500: {simple_regression.betas['SP500']:.3f}")
print(f"Beta VIX: {simple_regression.betas['VIX']:.3f}")

# Example 3: Risk decomposition
print("\nRisk decomposition:")
risk_decomp = regression_calc.calculate_risk_decomposition(
    simple_regression,
)
print(f"Systematic risk: {risk_decomp['systematic_pct']:.1f}%")
print(f"Idiosyncratic risk: {risk_decomp['idiosyncratic_pct']:.1f}%")

print("\nInsight: Components give you maximum flexibility for custom analysis")